# Solutions · 04 · Quasi-Monte Carlo

Worked solutions to the exercises of [notebook 04](../notebooks/04_quasi_monte_carlo.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engstoch is missing): install it from GitHub
    import engstoch
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engstoch"], check=True)
import math
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from engstoch import qmc, montecarlo as mc, brownian as bm, special as sf
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

R = np.random.default_rng      # R(seed): a fresh, reproducible generator

## Exercise 1

Price a European call by QMC: map Sobol points through the normal quantile and compare the error with MC for n = 2^8..2^16. Then do the same for a digital option (payoff 1 if S_T > K). Is the gain smaller, as one might expect for a discontinuous payoff?

In [2]:
S0, K, r, sig, T = 100.0, 100.0, 0.05, 0.2, 1.0
bs = bm.black_scholes_call(S0, K, r, sig, T)
d2 = (np.log(S0 / K) + (r - sig**2 / 2) * T) / (sig * np.sqrt(T))
digital = np.exp(-r * T) * sf.norm_cdf(d2)
def price(u, kind):
    z = sf.norm_ppf(np.clip(u, 1e-12, 1 - 1e-12))
    ST = S0 * np.exp((r - sig**2 / 2) * T + sig * np.sqrt(T) * z)
    return np.exp(-r * T) * (np.maximum(ST - K, 0) if kind == "call" else (ST > K).astype(float))
rows = []
for m in range(8, 17, 2):
    n = 2**m
    for kind, truth in (("call", bs), ("digital", digital)):
        e_mc = np.mean([abs(price(R(m * 10 + k).random(n), kind).mean() - truth) for k in range(10)])
        e_q = np.mean([abs(price(np.mod(qmc.van_der_corput(n, 2) + R(m * 100 + k).random(), 1), kind).mean() - truth) for k in range(10)])
        rows.append((n, kind, e_mc, e_q, e_mc / e_q))
print(pd.DataFrame(rows, columns=["n", "payoff", "MC error", "shifted-QMC error", "gain"]).to_string(index=False, float_format="%.2e"))

    n  payoff  MC error  shifted-QMC error     gain
  256    call  7.69e-01           1.13e-01 6.82e+00
  256 digital  2.45e-02           1.68e-03 1.46e+01
 1024    call  2.89e-01           4.17e-02 6.92e+00
 1024 digital  1.35e-02           1.29e-04 1.04e+02
 4096    call  2.43e-01           9.90e-03 2.45e+01
 4096 digital  5.28e-03           1.02e-04 5.18e+01
16384    call  8.73e-02           1.72e-03 5.09e+01
16384 digital  2.10e-03           1.62e-05 1.30e+02
65536    call  4.51e-02           7.04e-04 6.40e+01
65536 digital  1.77e-03           2.67e-06 6.63e+02


In one dimension the shifted van der Corput sequence is the 1-D Sobol sequence, and QMC's gain grows with n for
both payoffs. Surprisingly the digital option gains *more*: in one dimension a single jump has bounded
variation, so the Koksma-Hlawka bound still gives an O(log n / n) error, while the call's payoff, mapped through
the normal quantile, has a long unbounded tail near u = 1. The textbook trouble with discontinuities appears in
several dimensions, where the jump runs diagonally across the cube (a digital option on a basket or an Asian
average) and the integrand has infinite Hardy-Krause variation; conditioning out the jump restores QMC's rate.

## Exercise 2

For the integrand prod_j (1 + (x_j - 1/2)/j^2) in d = 20, compare MC and randomly shifted Sobol (use d <= 10 Sobol plus 10 Halton coordinates if you prefer). Explain the result with the idea of effective dimension.

In [3]:
d = 20
f = lambda x: np.prod(1 + (x - 0.5) / np.arange(1, d + 1) ** 2, axis=1)
def pts(n):
    return np.hstack([qmc.sobol(n, 10), qmc.halton(n, 10, start=1)])
rows = []
for m in (8, 10, 12, 14):
    n = 2**m
    e_mc = np.mean([abs(f(R(m * 10 + k).random((n, d))).mean() - 1) for k in range(10)])
    rq = qmc.randomised_qmc(f, pts(n), 10, R(m))
    rows.append((n, e_mc, np.mean(np.abs(rq["replicates"] - 1))))
print(pd.DataFrame(rows, columns=["n", "MC error", "randomised QMC error"]).to_string(index=False, float_format="%.2e"))
var_j = (1 / 12) / np.arange(1, d + 1) ** 4
print("share of the variance carried by the first 1, 2, 5 coordinates (first-order terms):", (np.cumsum(var_j) / var_j.sum())[[0, 1, 4]].round(4))

    n  MC error  randomised QMC error
  256  1.65e-02              1.03e-03
 1024  7.71e-03              3.27e-04
 4096  3.86e-03              1.38e-04
16384  1.35e-03              1.26e-04
share of the variance carried by the first 1, 2, 5 coordinates (first-order terms): [0.924  0.9817 0.9982]


The integral is exactly 1 and the nominal dimension is 20, yet QMC wins by orders of magnitude: the j-th
coordinate's influence decays like 1/j², so 92 % of the variance comes from the first coordinate alone. The
integrand has low *effective dimension*, and the first coordinates of Sobol points are the best distributed.
This is why QMC works for many high-dimensional problems in finance - and why ordering the inputs by importance
(for example by the Brownian-bridge construction) matters.

## Exercise 3

**Implement it yourself:** write the radical inverse function in any base with integer arithmetic and reproduce `qmc.halton(100, 5)` exactly.

In [4]:
PR = [2, 3, 5, 7, 11]
def radical_inverse_int(i, b):
    num, den = 0, 1
    while i > 0:
        i, d = divmod(i, b)
        num, den = num * b + d, den * b
    return num / den
H = np.array([[radical_inverse_int(i, b) for b in PR] for i in range(100)])
print("max difference from qmc.halton:", np.max(np.abs(H - qmc.halton(100, 5))))
print("first rows:\n", H[:4].round(4))

max difference from qmc.halton: 2.220446049250313e-16
first rows:
 [[0.     0.     0.     0.     0.    ]
 [0.5    0.3333 0.2    0.1429 0.0909]
 [0.25   0.6667 0.4    0.2857 0.1818]
 [0.75   0.1111 0.6    0.4286 0.2727]]


Accumulating the mirrored digits as an integer numerator over bᵏ and dividing once at the end avoids rounding
in the accumulation; the result agrees with the library to the last bit or two.